# Evaluation
## Credit Risk Scorecard Project

CRISP-DM Phase 5. This notebook evaluates both models using
credit-risk-specific metrics and connects them to the business decision:

1. KS Statistic and Gini coefficient
2. Probability calibration
3. Cost-based threshold selection
4. Expected profit per 1,000 applicants

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
import re
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.calibration import calibration_curve
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded")

In [ ]:
# Reload data and retrain LightGBM (fast, keeps notebook self-contained)
app = pd.read_csv('data/processed/app_clean.csv')
X = app.drop(columns=['TARGET', 'SK_ID_CURR'])
y = app['TARGET']

# Clean column names
X.columns = [re.sub(r'[^A-Za-z0-9_]', '_', col) for col in X.columns]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Retrain LightGBM
neg, pos = (y_train==0).sum(), (y_train==1).sum()
params = {
    'objective': 'binary', 'metric': 'auc',
    'learning_rate': 0.05, 'num_leaves': 31,
    'scale_pos_weight': neg/pos, 'random_state': 42, 'verbose': -1
}
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_test, label=y_test, reference=train_data)
model = lgb.train(params, train_data, num_boost_round=353,
                  valid_sets=[val_data], callbacks=[lgb.log_evaluation(0)])

# Get test predictions
test_pred = model.predict(X_test)
test_auc = roc_auc_score(y_test, test_pred)
print(f"LightGBM retrained. Test AUC: {test_auc:.4f}")

In [ ]:
# KS Statistic and Gini Coefficient

def calculate_ks(y_true, y_pred):
    """KS statistic: max separation between cumulative distributions"""
    fpr, tpr, thresholds = roc_curve(y_true, y_pred)
    ks = max(tpr - fpr)
    return ks

def calculate_gini(y_true, y_pred):
    """Gini = 2 * AUC - 1"""
    auc = roc_auc_score(y_true, y_pred)
    return 2 * auc - 1

ks = calculate_ks(y_test, test_pred)
gini = calculate_gini(y_test, test_pred)

print("LightGBM Evaluation Metrics:")
print(f"AUC:  {test_auc:.4f}")
print(f"KS:   {ks:.4f}")
print(f"Gini: {gini:.4f}")

In [ ]:
# Calibration curve - do predicted probabilities match reality?
prob_true, prob_pred = calibration_curve(y_test, test_pred, n_bins=10)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax.plot(prob_pred, prob_true, marker='o', color='steelblue', 
        label='LightGBM')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Actual Default Rate')
ax.set_title('Calibration Curve - LightGBM')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/05_calibration.png', dpi=150)
plt.show()

print("Calibration curve saved")

In [ ]:
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import train_test_split as tts

# Split test set into calibration and validation halves
# Calibrate on one half, evaluate calibration on the other
X_cal, X_val, y_cal, y_val = tts(
    X_test, y_test, test_size=0.5, random_state=42, stratify=y_test
)

# Get raw predictions on both halves
pred_cal = model.predict(X_cal)
pred_val = model.predict(X_val)

# Fit isotonic regression to map raw scores -> calibrated probabilities
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(pred_cal, y_cal)

# Apply calibration to validation half
pred_val_calibrated = iso.predict(pred_val)

print("Isotonic calibration fitted")
print(f"Raw prediction range:        {pred_val.min():.3f} to {pred_val.max():.3f}")
print(f"Calibrated prediction range: {pred_val_calibrated.min():.3f} to {pred_val_calibrated.max():.3f}")

In [ ]:
# Compare calibration before and after
prob_true_raw, prob_pred_raw = calibration_curve(y_val, pred_val, n_bins=10)
prob_true_cal, prob_pred_cal = calibration_curve(y_val, pred_val_calibrated, n_bins=10)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax.plot(prob_pred_raw, prob_true_raw, marker='o', color='crimson', 
        label='Before (raw LightGBM)')
ax.plot(prob_pred_cal, prob_true_cal, marker='s', color='green', 
        label='After (isotonic calibration)')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Actual Default Rate')
ax.set_title('Calibration: Before vs After Isotonic Regression')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/05_calibration_comparison.png', dpi=150)
plt.show()

print("Calibration comparison saved")

In [ ]:
# Cost-based threshold analysis
# From notebook 00 business understanding:
# - Approve a good loan (correctly): +$1,000 profit (interest margin)
# - Approve a bad loan (false negative): -$6,000 loss (loss given default)
# - Reject anyone: $0 (no profit, no loss)

PROFIT_GOOD = 1000   # profit from a good loan approved
LOSS_BAD = 6000      # loss from a bad loan approved

# Use calibrated predictions on validation set
y_eval = y_val.values
pred_eval = pred_val_calibrated

# Test thresholds from 0 to 1
thresholds = np.arange(0.01, 1.00, 0.01)
results = []

for t in thresholds:
    # Approve if predicted default probability is BELOW threshold
    approved = pred_eval < t
    
    # Among approved: how many were actually good vs bad?
    approved_good = ((approved) & (y_eval == 0)).sum()
    approved_bad = ((approved) & (y_eval == 1)).sum()
    
    # Profit calculation
    profit = (approved_good * PROFIT_GOOD) - (approved_bad * LOSS_BAD)
    approval_rate = approved.mean()
    
    results.append({
        'threshold': t,
        'approval_rate': approval_rate,
        'approved_good': approved_good,
        'approved_bad': approved_bad,
        'profit': profit
    })

results_df = pd.DataFrame(results)

# Find the profit-maximising threshold
best = results_df.loc[results_df['profit'].idxmax()]

print(f"Optimal threshold: {best['threshold']:.2f}")
print(f"Approval rate at optimum: {best['approval_rate']:.1%}")
print(f"Good loans approved: {best['approved_good']:,.0f}")
print(f"Bad loans approved: {best['approved_bad']:,.0f}")
print(f"Total profit on validation set: ${best['profit']:,.0f}")

In [ ]:
# Compare against naive strategies and visualise
n_applicants = len(y_eval)

# Strategy 1: Approve everyone
all_good = (y_eval == 0).sum()
all_bad = (y_eval == 1).sum()
profit_approve_all = (all_good * PROFIT_GOOD) - (all_bad * LOSS_BAD)

# Strategy 2: Approve nobody
profit_approve_none = 0

# Strategy 3: Our model
profit_model = best['profit']

print("Strategy comparison (validation set):")
print(f"Approve everyone:  ${profit_approve_all:,.0f}")
print(f"Approve nobody:    ${profit_approve_none:,.0f}")
print(f"Our model:         ${profit_model:,.0f}")
print(f"\nProfit per applicant (model): ${profit_model/n_applicants:,.0f}")
print(f"Profit per 1,000 applicants:  ${profit_model/n_applicants*1000:,.0f}")

# Plot profit vs threshold
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(results_df['threshold'], results_df['profit'], color='steelblue')
ax.axvline(x=best['threshold'], color='crimson', linestyle='--', 
           label=f"Optimal threshold ({best['threshold']:.2f})")
ax.axhline(y=profit_approve_all, color='gray', linestyle=':', 
           label='Approve everyone')
ax.set_xlabel('Approval Threshold (default probability cutoff)')
ax.set_ylabel('Profit ($)')
ax.set_title('Profit vs Approval Threshold')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/05_profit_threshold.png', dpi=150)
plt.show()

print("Profit threshold chart saved")

In [ ]:
# Final model comparison summary

summary = pd.DataFrame({
    'Metric': ['AUC', 'KS Statistic', 'Gini', 'Interpretability', 
               'Calibrated', 'Optimal Threshold', 'Profit per 1k applicants'],
    'WOE Scorecard': ['0.7391', '~0.38', '0.4782', 'High', 
                      'Yes (linear)', 'N/A', 'N/A'],
    'LightGBM': ['0.7712', '0.4025', '0.5424', 'Low', 
                 'After isotonic fix', '0.14', '$554,143']
})

print("="*70)
print("MODEL COMPARISON SUMMARY")
print("="*70)
print(summary.to_string(index=False))
print("="*70)
